<a href="https://colab.research.google.com/github/NicolasPetiot/EnjeuxDecarbonationSante/blob/main/contacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from scipy.spatial import distance_matrix

from pathlib import  Path

In [2]:
def load_pdb(path:Path) -> pd.DataFrame:
    """
    Parses a lines from a PDB file to extract DataFrame of atomic informations.
    """
    lines = path.read_text().splitlines()
    data  = [read_format(line) for line in lines if line.startswith(("ATOM", "HETATM"))]
    df = pd.DataFrame(data, columns = ["record_name", "name", "alt", "resn", "chain", "resi", "insertion", "x", "y", "z", "occupancy", "b", "segi", "e", "q"])
    return df

def read_format(line:str) -> tuple[str, str, str, str, int, str, float, float, float, float, float, float, str, str, str]:
    """
    Parses a line from a PDB file to extract atomic data.

    Args:
        line (str): A single line from the PDB file, formatted according
            to the PDB specification.

    Returns:
        tuple[str, str, str, str, int, str, float, float, float, float,
        float, float, str, str, str]: A tuple containing atom information,
        including:
            - Atom name
            - Alternate location indicator
            - Residue name
            - Chain identifier
            - Residue sequence number
            - Insertion code
            - X, Y, and Z coordinates
            - Occupancy and temperature factor
            - Segment identifier, element symbol, and charge
    """
    return (
        line[:6].strip(),
        #int(line[6:11].strip()),
        line[12:16].strip(),        # Atom name
        line[16:17].strip(),        # Alternate location indicator
        line[17:20].strip() ,       # Residue name
        line[21:22].strip(),        # Chain identifier
        int(line[22:26].strip()),   # Residue sequence number
        line[26:27].strip(),        # Insertion code
        float(line[30:38].strip()), # X coordinate
        float(line[38:46].strip()), # Y coordinate
        float(line[46:54].strip()), # Z coordinate
        float(line[54:60].strip()) if line[54:60].strip() != "" else 1.0, # Occupancy
        float(line[60:66].strip()) if line[60:66].strip() != "" else 0.0, # Temperature factor
        line[72:76].strip(),        # Segment identifier
        line[76:78].strip(),        # Element symbol
        line[78:80].strip()         # Charge on the atom
    )



In [3]:
filename = ""
path = Path(filename)
if not path.exists():
    raise FileNotFoundError(f"Le fichier '{filename}' n'est pas accessible ou n'existe pas")

df = load_pdb(path)

IsADirectoryError: [Errno 21] Is a directory: '.'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
protein = df.query("record_name == 'ATOM'")
ligand = df.query("resn == 'GSH'")
contact_threshold = 0.0 # TODO

xyz = ["x", "y", "z"] # Coordinate column selection
dists = distance_matrix(protein[xyz], ligand[xyz])
I, J = np.where(dists < contact_threshold)
protein.iloc[I]

In [ ]:
# PyMol selection string:
resi = protein.iloc[I].resi.unique().astype(str)
print("select Gsite, resi " + "+".join(resi))